In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_GetConnectionWorklist
# MAGIC Discovers eligible registered connections for one internally supplied
# MAGIC source system and publishes a safe Python-list worklist for a Databricks
# MAGIC For Each task. Supports CONFIGURED mode (pre-validation candidate discovery:
# MAGIC REGISTERED, VALID, FAILED requiring is_active=true; inactive rows ignored) and VALID mode
# MAGIC (post-validation operational discovery: VALID and is_active=true).
# MAGIC Metadata reads only: no adapter, secret, or JDBC access.

# COMMAND ----------

import json
from pyspark.sql import functions as F

try:
    from src.identifiers import escape_string_literal
    from src.source_identity import require_source_system, canonical_source_system_sql
    from src.worklist_utils import TASK_VALUE_LIMIT_BYTES, validate_task_value_payload
except ModuleNotFoundError:
    from identifiers import escape_string_literal
    from source_identity import require_source_system, canonical_source_system_sql
    from worklist_utils import TASK_VALUE_LIMIT_BYTES, validate_task_value_payload

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.dropdown("connection_mode", "VALID", ["VALID", "CONFIGURED"])
dbutils.widgets.text("catalog", "da_accelerators")
dbutils.widgets.text("control_schema", "control")
dbutils.widgets.text("max_connections", "0")
dbutils.widgets.text("only_connection_ids", "")
dbutils.widgets.text("exclude_connection_ids", "")

run_id = dbutils.widgets.get("run_id").strip()
source_system_raw = dbutils.widgets.get("source_system").strip()
connection_mode = (dbutils.widgets.get("connection_mode") or "VALID").strip().upper()
catalog = dbutils.widgets.get("catalog").strip()
control_schema = dbutils.widgets.get("control_schema").strip()

if not run_id:
    raise ValueError("run_id is required")
if not source_system_raw:
    raise ValueError("source_system is required")
if connection_mode not in ("VALID", "CONFIGURED"):
    raise ValueError(f"connection_mode must be 'VALID' or 'CONFIGURED', got {connection_mode!r}")
if not catalog:
    raise ValueError("catalog is required")
if not control_schema:
    raise ValueError("control_schema is required")

source_system = require_source_system(source_system_raw, "NB_GetConnectionWorklist")

try:
    max_connections = int(dbutils.widgets.get("max_connections").strip() or "0")
except ValueError as exc:
    raise ValueError("max_connections must be a non-negative integer") from exc
if max_connections < 0:
    raise ValueError("max_connections must be a non-negative integer")

raw_only = [
    value.strip() for value in
    dbutils.widgets.get("only_connection_ids").split(",") if value.strip()
]
if len(raw_only) != len(set(raw_only)):
    raise ValueError("only_connection_ids contains duplicate values")

raw_exclude = [
    value.strip() for value in
    dbutils.widgets.get("exclude_connection_ids").split(",") if value.strip()
]
if len(raw_exclude) != len(set(raw_exclude)):
    raise ValueError("exclude_connection_ids contains duplicate values")

clean_exclude = set(raw_exclude)
clean_only = {v for v in raw_only if v not in clean_exclude}

def _fqn(table_name):
    parts = (catalog, control_schema, table_name)
    return ".".join("`" + part.replace("`", "``") + "`" for part in parts)

connections = spark.table(_fqn("source_connection")).alias("sc")

eligible = (
    connections
    .filter(F.col("sc.connection_id").isNotNull())
    .filter(F.trim(F.col("sc.connection_id")) != "")
    .filter(F.col("sc.source_server").isNotNull() & (F.trim(F.col("sc.source_server")) != ""))
    .filter(F.col("sc.secret_scope").isNotNull() & (F.trim(F.col("sc.secret_scope")) != ""))
    .filter(F.expr(f"{canonical_source_system_sql('sc.source_system')} = {escape_string_literal(source_system)}"))
)

# SQL Server permits blank source_database for discover-all mode; Oracle requires it.
if source_system == "oracle":
    eligible = eligible.filter(F.col("sc.source_database").isNotNull() & (F.trim(F.col("sc.source_database")) != ""))

if connection_mode == "VALID":
    eligible = (
        eligible
        .filter(F.coalesce(F.col("sc.is_active"), F.lit(False)) == F.lit(True))
        .filter(F.upper(F.trim(F.col("sc.connection_status"))) == F.lit("VALID"))
    )
elif connection_mode == "CONFIGURED":
    eligible = (
        eligible
        .filter(F.coalesce(F.col("sc.is_active"), F.lit(False)) == F.lit(True))
        .filter(F.upper(F.trim(F.col("sc.connection_status"))).isin(["REGISTERED", "VALID", "FAILED"]))
    )

if raw_only and not clean_only:
    eligible = eligible.filter(F.lit(False))
elif clean_only:
    eligible = eligible.filter(F.col("sc.connection_id").isin(list(clean_only)))

if clean_exclude:
    eligible = eligible.filter(~F.col("sc.connection_id").isin(list(clean_exclude)))

eligible = (
    eligible.select(
        F.col("sc.connection_id").alias("connection_id")
    )
    .orderBy("connection_id")
)

if max_connections > 0:
    eligible = eligible.limit(max_connections)

rows = eligible.collect()

conn_ids = [row["connection_id"] for row in rows]
if len(conn_ids) != len(set(conn_ids)):
    raise ValueError("Connection worklist resolves to duplicate connection_id rows")

worklist = [
    {
        "connection_id": cid
    }
    for cid in conn_ids
]

for item in worklist:
    if set(item.keys()) != {"connection_id"}:
        raise ValueError(f"Worklist item contains unexpected keys: {list(item.keys())}")

validate_task_value_payload(worklist, key="worklist", limit_bytes=TASK_VALUE_LIMIT_BYTES)

worklist_count = len(worklist)
dbutils.jobs.taskValues.set(key="run_id", value=run_id)
dbutils.jobs.taskValues.set(key="source_system", value=source_system)
dbutils.jobs.taskValues.set(key="connection_mode", value=connection_mode)
dbutils.jobs.taskValues.set(key="worklist", value=worklist)
dbutils.jobs.taskValues.set(key="worklist_count", value=worklist_count)
print(f"Connection worklist: count={worklist_count}, source_system={source_system}, mode={connection_mode}")

if worklist_count == 0:
    exit_payload = {
        "status": "SUCCEEDED",
        "business_status": "NO_ELIGIBLE_CONNECTIONS",
        "run_id": run_id,
        "source_system": source_system,
        "connection_mode": connection_mode,
        "worklist_count": 0,
    }
else:
    exit_payload = {
        "status": "SUCCEEDED",
        "business_status": "READY",
        "run_id": run_id,
        "source_system": source_system,
        "connection_mode": connection_mode,
        "worklist_count": worklist_count,
        "worklist": worklist,
    }

dbutils.notebook.exit(json.dumps(exit_payload))
